# CV 4대 태스크 실습 — ②탐지 실습2: HOG 보행자 탐지

### 셀 1 — 패키지 설치 및 한글 폰트 설정

In [ ]:
!pip install -q opencv-python-headless matplotlib numpy
!apt-get -qq install -y fonts-nanum > /dev/null 2>&1

import abc
import os
import urllib.request
from dataclasses import dataclass
from typing import List

import cv2
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import numpy as np

nanum_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if os.path.exists(nanum_path):
    fm.fontManager.addfont(nanum_path)
    plt.rcParams["font.family"] = fm.FontProperties(fname=nanum_path).get_name()
plt.rcParams["axes.unicode_minus"] = False



### 셀 2 — [도메인 계층 / Domain Layer]

In [ ]:
# ----------------------------------------------------------------------------
# cv2에 의존하지 않는 순수 타입 정의. ex1과 동일한 DetectionBox/ObjectDetector
# 추상화를 재사용함으로써 OCP를 실제로 보여준다 — "사람"을 탐지하는 새로운
# 구현체(HOGPedestrianDetector)를 추가하는 데 도메인/유스케이스 계층은
# 전혀 수정하지 않는다.
# ===== [도메인 계층 / Domain Layer] =====

@dataclass
class DetectionBox:
    """탐지된 객체 하나를 나타내는 불변 값 객체(Value Object).

    x, y, w, h: 바운딩 박스 좌표(좌상단 기준) 및 크기
    label: 탐지된 객체의 종류 (예: "person")
    confidence: 탐지 신뢰도. HOG+SVM은 검출 점수(decision score)를 제공하므로
        그대로 채워 넣어 신뢰도 기반 후처리(NMS)에 활용한다.
    """

    x: int
    y: int
    w: int
    h: int
    label: str = "object"
    confidence: float = 1.0


class ObjectDetector(abc.ABC):
    """객체 탐지기의 추상 인터페이스(Port).

    ISP에 따라 "탐지" 책임만 가지며, 유스케이스 계층은 이 추상화에만
    의존한다(DIP). HOGPedestrianDetector는 이 인터페이스의 완전한
    대체재여야 한다(LSP) — 즉 어떤 입력에도 List[DetectionBox] 형태를
    예외 없이 반환해야 한다(빈 리스트 포함).
    """

    @abc.abstractmethod
    def detect(self, image: np.ndarray) -> List[DetectionBox]:
        """입력 이미지(BGR ndarray)에서 객체를 탐지해 DetectionBox 리스트로 반환한다."""
        raise NotImplementedError


class BoxSuppressor(abc.ABC):
    """중복 박스를 제거하는 후처리기의 추상 인터페이스(Port).

    탐지(Detection)와 후처리(NMS)는 서로 다른 책임이므로 별도 인터페이스로
    분리한다(SRP, ISP). 이렇게 분리해두면 NMS 알고리즘을 다른 방식(예:
    Soft-NMS)으로 교체해도 탐지기나 유스케이스를 건드릴 필요가 없다(OCP).
    """

    @abc.abstractmethod
    def suppress(self, boxes: List[DetectionBox]) -> List[DetectionBox]:
        """중복/저신뢰도 박스를 제거한 결과를 반환한다."""
        raise NotImplementedError



### 셀 3 — [유스케이스 계층 / Use Case Layer]

In [ ]:
# ----------------------------------------------------------------------------
# 탐지 → (선택적) NMS 후처리의 흐름을 조율한다. ObjectDetector와 BoxSuppressor
# 두 추상화 모두 생성자 주입으로 받으며, cv2를 직접 호출하지 않는다(DIP).
# ===== [유스케이스 계층 / Use Case Layer] =====

class DetectionUseCase:
    """보행자 탐지 + NMS 후처리 유스케이스.

    detector와 suppressor는 둘 다 추상 인터페이스 타입으로 주입받으므로,
    구체 구현(HOG, 다른 NMS 방식 등)을 교체해도 이 클래스는 수정되지
    않는다(OCP, DIP). suppressor가 주어지지 않으면 후처리 없이 원본
    탐지 결과를 그대로 반환한다.
    """

    def __init__(self, detector: ObjectDetector, suppressor: BoxSuppressor = None):
        self._detector = detector
        self._suppressor = suppressor

    def run(self, image: np.ndarray) -> List[DetectionBox]:
        """탐지를 실행하고, suppressor가 있으면 중복 제거까지 수행한다."""
        boxes = self._detector.detect(image)
        if self._suppressor is not None:
            boxes = self._suppressor.suppress(boxes)
        return boxes



### 셀 4 — [인프라/어댑터 계층 / Infrastructure Layer]

In [ ]:
# ----------------------------------------------------------------------------
# cv2를 실제로 호출하는 유일한 계층. HOG 탐지기 어댑터와 cv2.dnn.NMSBoxes를
# 사용하는 NMS 어댑터를 각각 별도 클래스로 둔다(SRP).
# ===== [인프라/어댑터 계층 / Infrastructure Layer] =====

class HOGPedestrianDetector(ObjectDetector):
    """HOG + 선형 SVM 기반 보행자 탐지 어댑터.

    얼굴 탐지(Haar Cascade)와 달리, 사람의 "전신 실루엣" 형태를 학습한
    OpenCV 내장 사전학습 모델(DefaultPeopleDetector)을 사용한다.
    ObjectDetector 인터페이스를 구현하므로 유스케이스 계층 입장에서는
    HaarCascadeFaceDetector와 완전히 동일하게 취급될 수 있다(LSP).
    """

    def __init__(self, win_stride=(8, 8), padding=(8, 8), scale: float = 1.05):
        self._hog = cv2.HOGDescriptor()
        self._hog.setSVMDetector(cv2.HOGDescriptor_getDefaultPeopleDetector())
        self._win_stride = win_stride
        self._padding = padding
        self._scale = scale

    def detect(self, image: np.ndarray) -> List[DetectionBox]:
        # detectMultiScale로 여러 크기의 사람을 탐색
        #  - winStride: 탐색 윈도우를 이동시키는 간격 (작을수록 정밀, 느림)
        #  - scale: 이미지 피라미드의 축소 비율
        boxes, weights = self._hog.detectMultiScale(
            image, winStride=self._win_stride, padding=self._padding, scale=self._scale,
        )
        return [
            DetectionBox(int(x), int(y), int(w), int(h), label="person", confidence=float(weight))
            for (x, y, w, h), weight in zip(boxes, weights.flatten() if len(weights) else [])
        ]


class NMSBoxSuppressor(BoxSuppressor):
    """cv2.dnn.NMSBoxes 기반 NMS(Non-Maximum Suppression) 어댑터.

    사람 1명에 여러 박스가 겹쳐 탐지되므로, 신뢰도가 낮거나 많이 겹치는
    박스를 정리한다. BoxSuppressor 인터페이스를 구현하므로 유스케이스는
    이 구체 구현을 알지 못한 채로 사용한다(DIP).
    """

    def __init__(self, score_threshold: float = 0.0, nms_threshold: float = 0.4):
        self._score_threshold = score_threshold
        self._nms_threshold = nms_threshold

    def suppress(self, boxes: List[DetectionBox]) -> List[DetectionBox]:
        if not boxes:
            return []
        raw_boxes = [[b.x, b.y, b.w, b.h] for b in boxes]
        scores = [b.confidence for b in boxes]
        idxs = cv2.dnn.NMSBoxes(
            bboxes=raw_boxes, scores=scores,
            score_threshold=self._score_threshold, nms_threshold=self._nms_threshold,
        )
        idxs = np.array(idxs).flatten() if len(idxs) > 0 else []
        return [boxes[i] for i in idxs]


class DetectionVisualizer:
    """탐지 결과를 이미지 위에 그리는 책임만 담당하는 클래스(SRP).

    탐지/후처리 로직과 분리되어 있어, 박스 색상이나 라벨 표기 형식을
    바꾸더라도 다른 계층에는 영향을 주지 않는다.
    """

    def __init__(self, color=(0, 140, 255), thickness: int = 2):
        self._color = color
        self._thickness = thickness

    def draw(self, image: np.ndarray, boxes: List[DetectionBox]) -> np.ndarray:
        result = image.copy()
        for box in boxes:
            cv2.rectangle(result, (box.x, box.y), (box.x + box.w, box.y + box.h),
                           self._color, self._thickness)
            cv2.putText(result, f"{box.label} {box.confidence:.2f}", (box.x, max(box.y - 8, 0)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, self._color, 2)
        return result



### 셀 5 — [구성 루트 / Composition Root — Colab 실행 코드]

In [ ]:
# ----------------------------------------------------------------------------
# 샘플 영상 준비 (OpenCV 공식 샘플 — 보행자가 걷는 거리 CCTV 영상)
# ===== [구성 루트 / Composition Root — Colab 실행 코드] =====

VIDEO_PATH = "/content/vtest.avi"
if not os.path.exists(VIDEO_PATH):
    url = "https://raw.githubusercontent.com/opencv/opencv/master/samples/data/vtest.avi"
    urllib.request.urlretrieve(url, VIDEO_PATH)
    print("샘플 영상 다운로드 완료:", VIDEO_PATH)

cap = cv2.VideoCapture(VIDEO_PATH)
cap.set(cv2.CAP_PROP_POS_FRAMES, 30)  # 사람이 잘 보이는 30번째 프레임 사용
ok, frame = cap.read()
cap.release()
if not ok:
    raise RuntimeError("영상 프레임을 읽지 못했습니다.")
print("프레임 크기:", frame.shape)


### 셀 6 — 구성 루트 — 어댑터 생성 및 유스케이스 주입(DI), 실행

In [ ]:
detector = HOGPedestrianDetector()
suppressor = NMSBoxSuppressor()
use_case = DetectionUseCase(detector, suppressor)

raw_boxes = detector.detect(frame)
print(f"1차 탐지된 박스 수: {len(raw_boxes)}")

final_boxes = use_case.run(frame)
print(f"NMS 이후 최종 탐지 수: {len(final_boxes)}")


### 셀 7 — 결과를 바운딩 박스로 시각화

In [ ]:
visualizer = DetectionVisualizer()
result = visualizer.draw(frame, final_boxes)
result_rgb = cv2.cvtColor(result, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(9, 7))
plt.imshow(result_rgb)
plt.title(f"HOG 보행자 탐지 결과 ({len(final_boxes)}명 탐지)")
plt.axis("off")
plt.tight_layout()
plt.savefig("/content/pedestrian_detection_result.png", dpi=120, bbox_inches="tight")
plt.show()
print("저장 완료: /content/pedestrian_detection_result.png")
